# Get Artifact 

In [1]:
# import wandb

# run = wandb.init(project="eb_jepa")
# artifact = run.use_artifact("hawardizayee-unitedhealthcare/eb_jepa/image_jepa_checkpoint:latest", type="model")
# artifact_dir = artifact.download()
# print(artifact_dir)  # prints where it was saved
# # The checkpoint will be downloaded to ./artifacts/image_jepa_checkpoint_v0/latest.pth.tar by default. You can then load it with:


In [2]:
artifact_dir = '/Users/hawardzaee/Desktop/AMI/eb_jepa/examples/my_image_jepa/artifacts/image_jepa_checkpoint:v0'

import torch
ckpt = torch.load(f"{artifact_dir}/latest.pth.tar", map_location="cpu")

# Get Val data 

In [3]:
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

from dataset import get_val_transforms

val_dataset = CIFAR10(
        root= "./data", train=False, download=False, transform=get_val_transforms()
    )

In [4]:
val_loader = DataLoader(val_dataset,batch_size=256,shuffle=False,drop_last=True)

In [5]:
for view,label in val_loader:
    print(view.shape,len(label))
    break


torch.Size([256, 3, 32, 32]) 256


# Model

In [6]:
from model import ResNet18, ImageSSL
from eval import LinearProbe

backbone = ResNet18()

model = ImageSSL(
    backbone= backbone,
    features_dim= backbone.features_dim
)

linear_probe = LinearProbe(feature_dim=backbone.features_dim,num_classes=10)


In [7]:
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

ImageSSL(
  (backbone): ResNet18(
    (backbone): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): Identity()
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_s

In [8]:
linear_probe.load_state_dict(ckpt["linear_probe_state_dict"])

<All keys matched successfully>

# Eval 

In [9]:
import torch.nn.functional as F 

In [10]:
with torch.no_grad():
    correct = 0 
    total_samples = 0 

    for data,target in val_loader:
        feature,z1 = model(data)

        # linear probing 
        pred = linear_probe(feature)

        correct += pred.argmax(1).eq(target).sum().item()
        total_samples += data.size(0)


    
 

In [11]:
(correct / total_samples) * 100.0

89.51322115384616

In [12]:
z1

tensor([[ 0.4084, -0.0511, -0.8389,  ...,  0.0602,  0.9925,  0.6545],
        [-0.6451, -0.8413, -0.9019,  ..., -0.3650, -0.0966, -0.1435],
        [-1.2455,  0.0898,  0.1138,  ..., -0.6220,  0.4836, -0.5382],
        ...,
        [-0.4799,  0.3007,  0.6469,  ..., -0.0280,  0.2340, -0.3635],
        [ 0.2336, -0.7982,  0.3290,  ...,  0.7254,  0.6599, -0.2055],
        [-0.1246,  0.0669, -0.5954,  ..., -0.7539, -0.1061,  0.7059]])